In [4]:
import json
import os
import pandas as pd
from chronos import Chronos2Pipeline


import time

start_time = time.time()

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"


countries = ["Germany","Ireland","Portugal"]
days = ["day1","day2","day3","day4","day5"]

countries = ["Denmark"]

days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
    "price_eur_kwh"
]

# 1 day ahead @ 15-min resolution
PRED_LEN = 96
QUANTILES = [0.1, 0.5, 0.9]
BATCH_SIZE = 128

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# LOAD SPLITS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# LOAD MODEL ONCE
# ============================================================
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

# ============================================================
# HELPERS
# ============================================================
def get_households(df: pd.DataFrame) -> list[str]:
    """Pick ONLY household columns and sort them home_1, home_2, ..."""
    homes = [c for c in df.columns if c.startswith("home_")]
    homes = sorted(homes, key=lambda x: int(x.split("_")[1]))
    return homes

def make_context_df(train_df: pd.DataFrame, households: list[str]) -> pd.DataFrame:
    """Wide -> long in Chronos format: timestamp, item_id, target"""
    return (
        train_df[households]
        .reset_index()
        .melt(id_vars=["timestamp"], var_name="item_id", value_name="target")
        .sort_values(["item_id", "timestamp"])
        .reset_index(drop=True)
    )

def to_wide_predictions(pred_long: pd.DataFrame, households: list[str]) -> pd.DataFrame:
    """
    Convert Chronos output long df -> wide df (timestamp index, homes as columns)
    Uses point forecast column 'predictions' if present (your case),
    otherwise falls back to median column '0.5' (if present).
    """
    if "predictions" in pred_long.columns:
        value_col = "predictions"
    elif "0.5" in pred_long.columns:
        value_col = "0.5"
    else:
        raise ValueError(f"Can't find point forecast column. Columns are: {list(pred_long.columns)}")

    wide = (
        pred_long
        .pivot(index="timestamp", columns="item_id", values=value_col)
        .sort_index()
    )

    # enforce column order and naming
    wide = wide.reindex(households, axis=1)
    wide.columns.name = None
    wide.index.name = "timestamp"
    return wide

# ============================================================
# MAIN LOOP
# ============================================================
for country in countries:
    print(f"\nProcessing country: {country}")

    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = get_households(df)
    if not households:
        raise ValueError(f"No home_* columns found in {data_path}. Columns: {list(df.columns)}")

    if country not in dataset_days:
        raise KeyError(f"{country} not found in dataset_days.json. Available: {list(dataset_days.keys())}")

    for day in days:
        print(f"  Day: {day}")

        if day not in dataset_days[country]:
            raise KeyError(f"{day} not found for {country} in dataset_days.json. Available: {list(dataset_days[country].keys())}")

        cutoff = pd.to_datetime(dataset_days[country][day])

        # train context up to cutoff
        train_df = df.loc[df.index < cutoff, households].copy()
        if train_df.empty:
            raise ValueError(f"Empty training slice for {country} {day}. cutoff={cutoff}")

        context_df = make_context_df(train_df, households)

        # Cross-learning joint prediction (ALL homes in one batch)
        pred_long = pipeline.predict_df(
            df=context_df,
            prediction_length=PRED_LEN,
            quantile_levels=QUANTILES,
            cross_learning=True,
            batch_size=BATCH_SIZE,
        )

        predictions_df_all_households = to_wide_predictions(pred_long, households)

        out_path = os.path.join(
            OUT_DIR,
            f"Chronos2_crosslearning_pred_{country.capitalize()}_{day}.csv"
        )
        predictions_df_all_households.to_csv(out_path, index=True, index_label="timestamp")

        print(f"    Saved: {out_path} | shape={predictions_df_all_households.shape}")


end_time = time.time()
total_seconds = end_time - start_time
print(f"Total runtime: {total_seconds:.2f} seconds")


Processing country: Denmark
  Day: day1
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_crosslearning_pred_Denmark_day1.csv | shape=(96, 9)
  Day: day2
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_crosslearning_pred_Denmark_day2.csv | shape=(96, 9)
  Day: day3
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_crosslearning_pred_Denmark_day3.csv | shape=(96, 9)
  Day: day4
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_crosslearning_pred_Denmark_day4.csv | shape=(96, 9)
  Day: day5
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_crosslearning_pred_Denmark_day5.csv | shape=(96, 9)
Total runtime: 2.53 seconds


In [5]:
print(f"Time taken: {total_seconds:.4f} seconds")


file_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\time_spend.json"
# 1. Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# 2. Add model inside "Local"
data["Foundational"]["Chronos2Global"] = total_seconds

# 3. Save back (without disturbing structure)
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

Time taken: 2.5337 seconds
